In [ ]:
# 02_etl_pipeline_notebook
# Databricks Notebook Source


# ETL Pipeline using PySpark

This notebook performs Extract, Transform, and Load (ETL)
operations on raw IoT sensor data and stores it in Delta format.


In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *


In [ ]:
sensor_schema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("event_ts", TimestampType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("voltage", DoubleType(), True)
])


In [ ]:
raw_df = (
    spark.read
    .schema(sensor_schema)
    .option("header", True)
    .csv("/Volumes/workspace/default/raw/raw_sensor_data.csv")
)

raw_df.show(5)


In [ ]:
clean_df = (
    raw_df
    .filter(col("sensor_id").isNotNull())
    .filter(col("voltage") >= 0)
    .dropDuplicates()
    .withColumn(
        "status",
        when(col("temperature") > 80, "Overheat").otherwise("Normal")
    )
)

clean_df.show(5)


In [ ]:
spark.sql(
    "CREATE DATABASE IF NOT EXISTS gold"
)
spark.sql(
    "DROP TABLE IF EXISTS gold.iot_sensor_gold"
)

(
    clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.iot_sensor_gold")
)

display(spark.table("gold.iot_sensor_gold"))